# Project 6 — Train a Mini GPT

## Overview
GPT (Generative Pre-trained Transformer) is a **decoder-only** transformer trained with the **next-token prediction** objective. We implement a character-level MiniGPT on the Tiny Shakespeare dataset.

**Key differences from Project 5 (encoder-style):**
- **Causal masking**: each token can only attend to past tokens
- **Learned positional embeddings** (not sinusoidal)
- **Training details**: AdamW, weight decay, learning rate warmup, gradient clipping

## 1. Theory — GPT Architecture

### Decoder-Only Transformer
GPT removes the cross-attention encoder-decoder interaction and uses only **self-attention with causal masking**. The causal mask ensures position $i$ cannot attend to positions $j > i$:
$$\text{mask}_{ij} = \begin{cases} 0 & \text{if } j \leq i \\ -\infty & \text{if } j > i \end{cases}$$

### Language Modelling Objective
Maximise the log-likelihood of each token given its context:
$$\mathcal{L} = -\frac{1}{T} \sum_{t=1}^{T} \log P(x_t \mid x_1, \ldots, x_{t-1})$$

### Perplexity
A common LM evaluation metric. Lower is better.
$$\text{PPL} = e^{\mathcal{L}} = \exp\left(-\frac{1}{T}\sum_{t=1}^T \log P(x_t|x_{<t})\right)$$

A model with PPL = $k$ is as uncertain as a uniform distribution over $k$ tokens.

### Generation Strategies

**Greedy**: always pick the most likely next token
$$\hat{x}_t = \arg\max_v P(v \mid x_{<t})$$

**Temperature sampling**: scale logits before softmax
$$P_\tau(v) = \frac{e^{z_v/\tau}}{\sum_u e^{z_u/\tau}}$$
- $\tau < 1$: more deterministic (peaked distribution)
- $\tau > 1$: more random (flatter distribution)

**Top-k sampling**: sample from only the top $k$ tokens
$$P(v) \propto \begin{cases} P_\tau(v) & \text{if } v \in \text{top-k} \\ 0 & \text{otherwise} \end{cases}$$

### AdamW Optimiser
AdamW decouples weight decay from the adaptive gradient computation:
$$\theta_{t+1} = \theta_t - \eta \left( \frac{m_t}{\sqrt{v_t} + \epsilon} + \lambda \theta_t \right)$$

This prevents the bias that standard Adam has when combining L2 regularisation with adaptive learning rates.

In [ ]:
import math
import urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 2. Dataset

In [ ]:
# Download tiny shakespeare (or use local copy)
URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'

try:
    import os
    if not os.path.exists('tinyshakespeare.txt'):
        print('Downloading Tiny Shakespeare...')
        urllib.request.urlretrieve(URL, 'tinyshakespeare.txt')
    with open('tinyshakespeare.txt', 'r') as f:
        TEXT = f.read()
    print(f'Loaded {len(TEXT):,} characters from file.')
except Exception as e:
    print(f'Download failed ({e}). Using embedded excerpt.')
    # Fallback embedded text
    TEXT = """HAMLET:\nTo be, or not to be, that is the question:\nWhether 'tis nobler in the mind to suffer\nThe slings and arrows of outrageous fortune,\nOr to take arms against a sea of troubles\nAnd by opposing end them.\n\nOPHELIA:\nO, what a noble mind is here o'erthrown!\nThe courtier's, soldier's, scholar's, eye, tongue, sword;\nThe expectancy and rose of the fair state,\nThe glass of fashion and the mould of form,\n""" * 200

# Character tokeniser
chars     = sorted(set(TEXT))
vocab_size = len(chars)
stoi      = {c: i for i, c in enumerate(chars)}
itos      = {i: c for c, i in stoi.items()}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data   = torch.tensor(encode(TEXT), dtype=torch.long)
n      = len(data)
n_train = int(0.9 * n)
train_data = data[:n_train]
val_data   = data[n_train:]

print(f'Vocab size:   {vocab_size}')
print(f'Total tokens: {n:,}')
print(f'Train: {n_train:,} | Val: {n - n_train:,}')

## 3. MiniGPT Architecture

In [ ]:
class GPTConfig:
    vocab_size  : int
    block_size  : int = 128   # context length
    n_embd      : int = 128   # embedding dimension
    n_head      : int = 4
    n_layer     : int = 4
    dropout     : float = 0.1


class CausalSelfAttention(nn.Module):
    """Multi-head self-attention with causal (autoregressive) mask."""

    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0

        self.n_head   = n_head
        self.n_embd   = n_embd
        self.head_dim = n_embd // n_head

        # Combined Q, K, V projection (3x for efficiency)
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

        # Causal mask: lower triangular matrix of ones
        # (1, 1, T, T) to broadcast over batch and heads
        mask = torch.tril(torch.ones(block_size, block_size))
        self.register_buffer('mask', mask.view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.size()

        # Compute Q, K, V
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)  # each (B, T, n_embd)

        # Reshape to (B, n_head, T, head_dim)
        def reshape(t):
            return t.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        q, k, v = reshape(q), reshape(k), reshape(v)

        # Scaled dot-product attention
        attn = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))

        # Apply causal mask: fill future positions with -inf before softmax
        attn = attn.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        attn = self.attn_drop(attn)

        # Aggregate values
        out = attn @ v  # (B, n_head, T, head_dim)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.c_proj(out))


class GPTBlock(nn.Module):
    """GPT transformer block with Pre-LayerNorm."""

    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln_1  = nn.LayerNorm(n_embd)
        self.attn  = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln_2  = nn.LayerNorm(n_embd)
        self.mlp   = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))   # attention with residual
        x = x + self.mlp(self.ln_2(x))    # feed-forward with residual
        return x


class MiniGPT(nn.Module):
    """Character-level GPT."""

    def __init__(self, vocab_size, block_size=128, n_embd=128, n_head=4, n_layer=4, dropout=0.1):
        super().__init__()
        self.block_size = block_size

        self.transformer = nn.ModuleDict({
            'wte': nn.Embedding(vocab_size, n_embd),           # token embedding
            'wpe': nn.Embedding(block_size, n_embd),           # position embedding (learned)
            'drop': nn.Dropout(dropout),
            'h': nn.ModuleList([GPTBlock(n_embd, n_head, block_size, dropout) for _ in range(n_layer)]),
            'ln_f': nn.LayerNorm(n_embd),                      # final layer norm
        })
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

        # Weight tying: output head shares weights with token embedding
        self.transformer['wte'].weight = self.lm_head.weight

        # Initialise weights
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.block_size

        pos   = torch.arange(T, device=idx.device)  # [0, 1, ..., T-1]
        tok_emb = self.transformer['wte'](idx)       # (B, T, n_embd)
        pos_emb = self.transformer['wpe'](pos)       # (T, n_embd) — broadcast over batch

        x = self.transformer['drop'](tok_emb + pos_emb)
        for block in self.transformer['h']:
            x = block(x)
        x = self.transformer['ln_f'](x)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """Autoregressive generation with optional top-k sampling."""
        self.eval()
        for _ in range(max_new_tokens):
            # Crop to block_size
            idx_cond = idx if idx.size(1) <= self.block_size else idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits    = logits[:, -1, :] / temperature  # (B, vocab_size)

            if top_k is not None:
                # Zero out all logits except top-k
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float('-inf')

            probs    = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx      = torch.cat([idx, next_idx], dim=1)

        return idx


# Hyperparameters
BLOCK_SIZE = 128
N_EMBD     = 128
N_HEAD     = 4
N_LAYER    = 4
DROPOUT    = 0.1
BATCH_SIZE = 32

model = MiniGPT(
    vocab_size=vocab_size,
    block_size=BLOCK_SIZE,
    n_embd=N_EMBD,
    n_head=N_HEAD,
    n_layer=N_LAYER,
    dropout=DROPOUT,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'MiniGPT parameters: {n_params:,}')

## 4. Training with LR Warmup

In [ ]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix   = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,))
    x    = torch.stack([data[i  : i+BLOCK_SIZE  ] for i in ix])
    y    = torch.stack([data[i+1: i+BLOCK_SIZE+1] for i in ix])
    return x.to(device), y.to(device)


# AdamW with weight decay
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.95),
    weight_decay=0.1
)

# Learning rate warmup scheduler
WARMUP_STEPS = 100
MAX_STEPS    = 2000

def get_lr(step):
    """Linear warmup then cosine decay."""
    if step < WARMUP_STEPS:
        return step / WARMUP_STEPS         # linear warmup
    # Cosine decay after warmup
    progress = (step - WARMUP_STEPS) / (MAX_STEPS - WARMUP_STEPS)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, get_lr)

@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ['train', 'val']:
        losses = []
        for _ in range(20):
            xb, yb  = get_batch(split)
            _, loss = model(xb, yb)
            losses.append(loss.item())
        out[split] = np.mean(losses)
    model.train()
    return out


train_hist, val_hist, ppl_hist = [], [], []
generated_samples = {}  # store samples at different checkpoints

print(f'Training MiniGPT for {MAX_STEPS} steps...')
model.train()

for step in range(1, MAX_STEPS + 1):
    xb, yb   = get_batch('train')
    _, loss  = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
    optimizer.step()
    scheduler.step()

    if step % 200 == 0 or step == 1:
        losses = estimate_loss()
        ppl    = math.exp(losses['val'])
        train_hist.append(losses['train'])
        val_hist.append(losses['val'])
        ppl_hist.append(ppl)
        lr = scheduler.get_last_lr()[0] * 3e-4
        print(f'Step {step:>5}  train={losses["train"]:.4f}  val={losses["val"]:.4f}  ppl={ppl:.1f}  lr={lr:.2e}')

        # Generate sample
        ctx = torch.tensor([[stoi['\n']]], dtype=torch.long, device=device)
        gen = model.generate(ctx, max_new_tokens=100, temperature=0.8, top_k=10)
        generated_samples[step] = decode(gen[0].tolist())

print('\nTraining complete!')

In [ ]:
# Plot training metrics
checkpoints = [1] + list(range(200, MAX_STEPS + 1, 200))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(checkpoints, train_hist, label='Train')
axes[0].plot(checkpoints, val_hist,   label='Val')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training/Validation Loss')
axes[0].legend()

axes[1].plot(checkpoints, ppl_hist, color='darkorange')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Perplexity')
axes[1].set_title('Validation Perplexity')

plt.tight_layout()
plt.show()

In [ ]:
# Show generation at different checkpoints
print('=== Text Generation at Different Checkpoints ===\n')
for step, text in list(generated_samples.items())[-3:]:
    print(f'--- Step {step} ---')
    print(text[:200])
    print()

# Final generation with longer output
print('\n=== Final Model — 500 character generation ===' )
ctx = torch.tensor([[stoi['\n']]], dtype=torch.long, device=device)
gen = model.generate(ctx, max_new_tokens=500, temperature=0.8, top_k=10)
print(decode(gen[0].tolist()))

## Summary

| Feature | Implementation |
|---|---|
| Architecture | 4-layer decoder-only transformer |
| Attention | Causal (masked) self-attention |
| Position | Learned position embeddings |
| Optimiser | AdamW with weight decay |
| LR Schedule | Linear warmup + cosine decay |
| Regularisation | Dropout + gradient clipping |
| Generation | Top-k sampling with temperature |

**Key insight**: The causal mask is what makes GPT an **autoregressive** model — it can only see past context, which means it can generate text token by token without any special inference mechanism.